# Exercice 2 — Segmenter, visualiser et surveiller un portefeuille vie (+ mini apprentissage par renforcement)

**Le contexte.** Le directeur du portefeuille veut une *photographie lisible* de ses ~3 500 contrats : quels profils, comment les montrer en une page au comité, et quels contrats « bizarres » faire revoir par un gestionnaire. **Il n'y a pas de cible** : on explore.

| # | Étape | Notion de la session |
|---|---|---|
| Q1 | Pourquoi standardiser | distance, dépendance à l'échelle |
| Q2 | Choisir k | silhouette, stabilité, sens métier |
| Q3 | Nommer les segments | grille d'interprétation à 4 colonnes |
| Q4 | PCA | variance expliquée, loadings, perte d'information |
| Q5 | Un cluster porté par une seule variable | piège du non supervisé |
| Q6 | Anomalies | distance au centre, Isolation Forest, revue experte |
| Q7 | Mini Q-learning | agent, récompense, γ, politique |
| Q8 | Quiz de consolidation | quand utiliser / éviter |

> **Mode d'emploi.** Remplacez chaque `...` par votre code, exécutez la cellule `verifier(...)` suivante. Les 💡 indices sont repliables.

## Préparation (cellule fournie — à exécuter)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

import hashlib
RES = {}
def verifier(qid, test, ok="Bravo !", ko="Pas encore — relis l'énoncé ou ouvre l'indice."):
    """Auto-correction : exécute `test()` et enregistre le résultat de la question `qid`."""
    try:
        res = bool(test()); detail = ""
    except Exception as e:
        res = False; detail = f" [{type(e).__name__} : {e}]"
    RES[qid] = res
    print(("✅ " + ok) if res else ("❌ " + ko + detail))

def qcm(qid, reponse, empreinte):
    ok = hashlib.sha256(f"{qid}:{str(reponse).strip().upper()}".encode()).hexdigest()[:10] == empreinte
    RES[qid] = ok
    print("✅ Bonne réponse !" if ok else "❌ Ce n'est pas la bonne réponse — relis la slide correspondante.")

def bilan(attendus):
    print("BILAN DE L'EXERCICE")
    for q in attendus: print(f"  {q:<6}", "✅" if RES.get(q) else "⬜ à faire / à revoir")
    print(f"\nScore : {sum(RES.get(q, False) for q in attendus)}/{len(attendus)}")
import numpy as np, pandas as pd

def generer_portefeuille_seg(seed=7):
    """Portefeuille vie synthétique (4 profils latents + quelques contrats atypiques injectés)."""
    rng = np.random.default_rng(seed)
    def bloc(n, age, anc, enc, vers, arb, rach):
        a = np.clip(rng.normal(*age, n), 20, 90).round()
        e = np.exp(rng.normal(*enc, n)).round(-2)
        return pd.DataFrame(dict(age=a, anciennete=np.clip(rng.normal(*anc, n), 0.2, 30).round(1), encours=e,
            versements_12m=np.where(rng.random(n) < vers[0], e*rng.uniform(.03, vers[1], n), 0).round(-1),
            nb_arbitrages_12m=rng.poisson(arb, n), nb_rachats_passes=rng.poisson(rach, n)))
    df = pd.concat([bloc(1100, (34,5), (2.5,1.3), (9.6,.5), (.9,.25), 1.5, .3),
                    bloc(800,  (62,7), (10,4),   (12.0,.5), (.2,.08), .2, .05),
                    bloc(700,  (52,9), (8.3,.6), (10.8,.6), (.05,.06), .5, .8),
                    bloc(900,  (71,6), (15,5),   (10.4,.6), (.02,.06), .05, .1)], ignore_index=True)
    ano = pd.DataFrame(dict(
        age=[30]*12 + [58]*10 + [49]*8, anciennete=[27 + i*.2 for i in range(12)] + [9]*10 + [6]*8,
        encours=[0]*12 + [1_200_000 + 60_000*i for i in range(10)] + [30_000]*8,
        versements_12m=[0]*12 + [0]*10 + [250_000 + 20_000*i for i in range(8)],
        nb_arbitrages_12m=[0]*12 + [8]*10 + [1]*8, nb_rachats_passes=[0]*12 + [7]*10 + [0]*8))
    df = pd.concat([df, ano], ignore_index=True)
    ids_ano = set(range(len(df) - len(ano), len(df)))
    idx = rng.permutation(len(df))                       # mélange : la position ne trahit pas les anomalies
    df = df.iloc[idx].reset_index(drop=True)
    df.insert(0, "contrat_id", np.arange(1, len(df) + 1))
    ids = {int(i) + 1 for i, old in enumerate(idx) if old in ids_ano}
    return df, ids

df, _IDS_ANOMALIES = generer_portefeuille_seg()      # _IDS_ANOMALIES : vérité cachée, utilisée seulement par les vérifications
VARS = ["age", "anciennete", "encours", "versements_12m", "nb_arbitrages_12m", "nb_rachats_passes"]
print(f"{len(df):,} contrats, {len(VARS)} variables, aucune cible.")
df.head()

### Dictionnaire de données
| Variable | Description |
|---|---|
| `age` | Âge de l'assuré (années) |
| `anciennete` | Ancienneté du contrat (années) |
| `encours` | Provision mathématique (€) — distribution très asymétrique |
| `versements_12m` | Versements des 12 derniers mois (€) |
| `nb_arbitrages_12m` | Nombre d'arbitrages sur 12 mois |
| `nb_rachats_passes` | Nombre de rachats passés |

Pour que les euros ne dominent pas, `encours` et `versements_12m` sont passés en `log1p` (fourni ci-dessous) : les variables en € suivent une loi très asymétrique.

## Q1 · Pourquoi standardiser avant un k-means ?
**Notion :** *sans standardisation, l'encours écrase l'âge dans la distance : les clusters suivent l'euro, pas le comportement.*

1. Créez `X_std` : `StandardScaler().fit_transform(...)` appliqué à `X_log` (variables `VARS`, avec `encours` et `versements_12m` en log).
2. Entraînez deux k-means (`n_clusters=4, n_init=10, random_state=0`) : `km_brut` sur `df[VARS]` en euros bruts, et `km` sur `X_std`.
3. La fonction `part_variance_encours(labels)` (fournie) donne la part de la variance de l'`encours` expliquée par les clusters. Comparez `eta_brut` et `eta_std`.


<details><summary>💡 <b>Indice</b></summary>

`km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)` ; les étiquettes sont dans `km.labels_`.

</details>

In [ ]:
X_log = df[VARS].copy()
X_log["encours"] = np.log1p(X_log["encours"]); X_log["versements_12m"] = np.log1p(X_log["versements_12m"])

def part_variance_encours(labels):
    e = df["encours"]; labels = np.asarray(labels)
    intra = sum(((e[labels == c] - e[labels == c].mean()) ** 2).sum() for c in np.unique(labels))
    return 1 - intra / ((e - e.mean()) ** 2).sum()

X_std = ...      # TODO
km_brut = ...    # TODO : KMeans sur df[VARS] (euros bruts)
km = ...         # TODO : KMeans sur X_std
eta_brut = part_variance_encours(km_brut.labels_)
eta_std = part_variance_encours(km.labels_)
print(f"Part de variance de l'encours expliquée : bruts {eta_brut:.0%} | standardisés {eta_std:.0%}")

In [ ]:
verifier("Q1", lambda: abs(X_std.mean()) < 1e-6 and abs(X_std.std() - 1) < 1e-6 and eta_brut > 0.8 and eta_std < 0.5,
         "Sur données brutes, les clusters ne sont qu'un découpage de l'encours (>80 % de sa variance) ; standardisés, ils reflètent plusieurs dimensions.",
         "X_std doit être centrée-réduite, et km_brut/km doivent être des KMeans(4) entraînés sur les bonnes données.")

## Q2 · Choisir k : silhouette, stabilité, sens métier
**Notion :** *la silhouette propose, le métier dispose.*

1. Pour `k` de 2 à 8, calculez la **silhouette moyenne** de `KMeans(k, n_init=10, random_state=0)` sur `X_std` → dictionnaire `sil`. `k_geo` = k qui la maximise.
2. **Stabilité** (k = 4) : pour 5 graines `b` (0 à 4), tirez 80 % des contrats sans remise (`rng = np.random.default_rng(b)`, `rng.choice(len(X_std), int(0.8*len(X_std)), replace=False)`), entraînez un k-means sur cet échantillon, prédisez les étiquettes de **tous** les contrats et calculez l'`adjusted_rand_score` contre `km.labels_`. `ari_moyen` = moyenne.
3. Quel k retenez-vous ? `K_METIER` (le choix documenté : ici, on veut isoler le profil « pré-échéance fiscale », donc **4**).


<details><summary>💡 <b>Indice</b></summary>

`silhouette_score(X_std, labels)` ; ARI proche de 1 = segments qui survivent au rééchantillonnage.

</details>

In [ ]:
sil = {}
for k in range(2, 9):
    ...   # TODO : sil[k] = silhouette moyenne
k_geo = ...   # TODO

aris = []
for b in range(5):
    rng = np.random.default_rng(b)
    idx = ...   # TODO : 80 % des indices sans remise
    km_b = ...  # TODO : KMeans(4) sur X_std[idx]
    aris.append(adjusted_rand_score(km.labels_, km_b.predict(X_std)))
ari_moyen = np.mean(aris)
K_METIER = ...  # TODO
print({k: round(v, 3) for k, v in sil.items()}, "| k géométrique :", k_geo, "| ARI moyen :", round(ari_moyen, 3))

In [ ]:
verifier("Q2a", lambda: set(sil) == set(range(2, 9)) and k_geo == max(sil, key=sil.get) and 0 < sil[2] < 1, "Balayage de k correct.")
verifier("Q2b", lambda: ari_moyen > 0.8, "Segments stables au rééchantillonnage (ARI proche de 1).")
verifier("Q2c", lambda: K_METIER == 4, "k = 4 : choix métier assumé, même si k_geo (silhouette) est plus petit — à documenter comme un choix, pas un résultat automatique.",
         "Relisez l'énoncé : on veut isoler le segment « pré-échéance fiscale ».")
plt.plot(list(sil), list(sil.values()), "o-"); plt.axvline(K_METIER, ls=":", c="r"); plt.xlabel("k"); plt.ylabel("silhouette"); plt.title("Silhouette moyenne selon k"); plt.show()

## Q3 · Profiler et nommer les segments
**Notion :** *un cluster qu'on ne peut pas nommer en langage métier est inutilisable.*

1. `profil` : moyenne des 6 variables (en unités d'origine) par cluster de `km`, avec la colonne `effectif`.
2. `profil_z` : écart de chaque moyenne de cluster à la moyenne du portefeuille, en nombre d'écarts-types → repérez les variables dominantes.
3. Donnez un **nom** à chaque cluster dans `NOMS` (dictionnaire `{numéro: "nom"}`) et complétez la grille à 4 colonnes en Markdown plus bas.


<details><summary>💡 <b>Indice</b></summary>

- `df.assign(cluster=km.labels_).groupby("cluster")[VARS].mean()`
- `profil_z = (profil[VARS] - df[VARS].mean()) / df[VARS].std()`

</details>

In [ ]:
dfc = df.assign(cluster=km.labels_)
profil = ...     # TODO (colonnes VARS) ; ajoutez ensuite la colonne "effectif"
profil_z = ...   # TODO
NOMS = {0: "...", 1: "...", 2: "...", 3: "..."}   # TODO

In [ ]:
display(profil.round(1)); display(profil_z.round(2))
verifier("Q3", lambda: set(NOMS) == set(profil.index) and len(set(NOMS.values())) == 4 and all(len(str(v)) > 3 and v != "..." for v in NOMS.values())
         and profil_z.shape == (4, 6),
         "4 segments, 4 noms distincts. Vérifiez que chacun se raconte en une phrase.")

**Grille d'interprétation — à compléter :**

| Segment | Variables dominantes | Usage potentiel | Risque de mauvaise lecture |
|---|---|---|---|
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |

## Q4 · PCA : projeter pour voir, en assumant la perte
**Notion :** *la PCA conserve de la variance, pas du sens.*

1. `pca = PCA().fit(X_std)`. Calculez `var_cumulee` (variance expliquée cumulée) et `n_80` : nombre minimal de composantes pour expliquer au moins **80 %** de la variance.
2. `loadings` : DataFrame des coefficients (`pca.components_.T`) des 2 premières composantes, indexé par `VARS`, colonnes `["PC1", "PC2"]`.
3. Lisez PC1 : quelles variables s'opposent ? Sous quelle étiquette métier la décririez-vous ?


<details><summary>💡 <b>Indice</b></summary>

`np.cumsum(pca.explained_variance_ratio_)` ; `n_80 = int(np.argmax(var_cumulee >= 0.80)) + 1`.

</details>

In [ ]:
pca = ...           # TODO
var_cumulee = ...   # TODO
n_80 = ...          # TODO
loadings = ...      # TODO
Z = pca.transform(X_std)[:, :2]

In [ ]:
verifier("Q4", lambda: abs(var_cumulee[-1] - 1) < 1e-9 and n_80 == int(np.argmax(np.cumsum(PCA().fit(X_std).explained_variance_ratio_) >= 0.8)) + 1
         and loadings.shape == (6, 2), f"{n_80} composantes suffisent pour 80 % de variance : la projection 2D en conserve {var_cumulee[1]:.0%} (le reste est perdu).")
display(loadings.round(2))
plt.scatter(Z[:, 0], Z[:, 1], c=km.labels_, s=6, cmap="tab10"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Portefeuille en 2D (couleur = cluster)"); plt.show()

## Q5 · Piège : un cluster porté par une seule variable
**Notion (exercice facultatif de la session) :** *retirez `anciennete` et observez le segment « pré-échéance fiscale » disparaître.*

1. Refaites tout le pipeline **sans** `anciennete` : `X_log_sans`, `X_std_sans` (nouveau `StandardScaler`), `km_sans` (k = 4, mêmes réglages).
2. Calculez `ari_sans` = ARI entre `km.labels_` et `km_sans.labels_`.
3. Interprétez : l'ARI vaut 1 si les deux segmentations sont identiques ; quelle part de la structure repose sur `anciennete` ?


<details><summary>💡 <b>Indice</b></summary>

`adjusted_rand_score(km.labels_, km_sans.labels_)` ; un ARI < 0,8 signale une segmentation qui change nettement.

</details>

In [ ]:
X_log_sans = ...   # TODO : X_log sans la colonne "anciennete"
X_std_sans = ...   # TODO
km_sans = ...      # TODO
ari_sans = ...     # TODO
print(f"ARI avec/sans ancienneté : {ari_sans:.2f}")

In [ ]:
verifier("Q5", lambda: X_log_sans.shape[1] == 5 and abs(ari_sans - adjusted_rand_score(km.labels_, km_sans.labels_)) < 1e-9 and ari_sans < 0.8,
         "La segmentation change nettement sans l'ancienneté : une partie de la structure reposait sur une seule variable.")

## Q6 · Anomalies : prioriser l'attention humaine
**Notion :** *score = distance au plus proche centre ; Isolation Forest ; le score classe par étrangeté, pas par importance.*

1. `dist` : pour chaque contrat, distance au centre le plus proche (`km.transform(X_std).min(axis=1)`). `top_km` = positions des **30** contrats les plus éloignés.
2. `iso = IsolationForest(n_estimators=200, random_state=0).fit(X_std)` ; `score_iso = -iso.score_samples(X_std)` (plus grand = plus atypique). `top_iso` = positions des 30 plus atypiques.
3. `recoupement` : nombre de contrats communs aux deux listes.


<details><summary>💡 <b>Indice</b></summary>

`np.argsort(-score)[:30]` donne les 30 positions aux plus grands scores.

</details>

In [ ]:
dist = ...        # TODO
top_km = ...      # TODO
score_iso = ...   # TODO
top_iso = ...     # TODO
recoupement = ... # TODO
print("Contrats communs aux deux méthodes :", recoupement, "/ 30")

In [ ]:
def _vrais(top): return len({int(df["contrat_id"].iloc[i]) for i in top} & _IDS_ANOMALIES)
verifier("Q6a", lambda: len(top_km) == 30 and _vrais(top_km) >= 15, f"Score de distance : {_vrais(top_km)}/30 des contrats signalés sont des anomalies injectées.")
verifier("Q6b", lambda: len(top_iso) == 30 and _vrais(top_iso) >= 15, f"Isolation Forest : {_vrais(top_iso)}/30 des contrats signalés sont des anomalies injectées.")
verifier("Q6c", lambda: recoupement == len(set(top_km) & set(top_iso)) and recoupement >= 10, "Les deux méthodes convergent largement : le consensus est un bon candidat à la revue experte.")
df.iloc[top_km[:8]][["contrat_id"] + VARS]

## Q7 · Mini apprentissage par renforcement : le monde-grille actuariel
**Notions :** agent · environnement · récompense · politique · retour actualisé (γ) · Q-learning · exploration / exploitation.

Chaque case de la grille 4 × 4 est un **état de marché**, chaque mouvement (↑ ↓ ← →) une **décision d'allocation**. Départ en haut à gauche ; les cases rouges `R` sont des **chocs de taux** (récompense −10, fin d'épisode), la case verte `G` l'objectif de rendement ajusté du risque (+10, fin). Chaque pas coûte −1 (coût de portage / de liquidité).

**7a — Retour actualisé.** Écrivez `retour(recompenses, gamma)` = Σ γᵗ·rₜ (ce n'est autre qu'une valeur actuelle avec γ comme facteur d'actualisation).

**7b — Mise à jour de Q.** Dans la boucle, écrivez la règle de Q-learning :
`Q[s, a] ← Q[s, a] + α · ( r + γ · max_a' Q[s', a'] − Q[s, a] )` (sans le terme `max` si l'épisode est terminé).


<details><summary>💡 <b>Indice</b></summary>

Si `fini` : `cible = r` ; sinon `cible = r + gamma * Q[s2].max()`. Puis `Q[s, a] += alpha * (cible - Q[s, a])`.

</details>

In [ ]:
def retour(recompenses, gamma):
    ...   # TODO

MOUVEMENTS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # ↑ ↓ ← →
CHOCS, OBJECTIF, N = {(1, 1), (2, 3)}, (3, 3), 4

def pas(etat, a):
    """Environnement (fourni) : renvoie (nouvel état, récompense, fini)."""
    i, j = etat[0] + MOUVEMENTS[a][0], etat[1] + MOUVEMENTS[a][1]
    nouveau = (min(max(i, 0), N - 1), min(max(j, 0), N - 1))
    if nouveau in CHOCS: return nouveau, -10, True
    if nouveau == OBJECTIF: return nouveau, 10, True
    return nouveau, -1, False

def entrainer(episodes=3000, alpha=0.1, gamma=0.95, eps=0.2, graine=0):
    rng = np.random.default_rng(graine)
    Q = np.zeros((N, N, 4))
    for _ in range(episodes):
        s, fini, t = (0, 0), False, 0
        while not fini and t < 100:
            a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s]))    # ε-greedy : exploration / exploitation
            s2, r, fini = pas(s, a)
            ...   # TODO : mise à jour de Q[s][a]
            s, t = s2, t + 1
    return Q

In [ ]:
verifier("Q7a", lambda: abs(retour([1, 1, 1], 0.5) - 1.75) < 1e-9 and abs(retour([5, 5], 1.0) - 10) < 1e-9,
         "Retour actualisé correct : γ = 0,5 pondère très fortement le présent (1 + 0,5 + 0,25 = 1,75).")
Q = entrainer()

def politique_gloutonne(Q):
    s, chemin, recomp = (0, 0), [(0, 0)], []
    for _ in range(20):
        s, r, fini = pas(s, int(np.argmax(Q[s]))); chemin.append(s); recomp.append(r)
        if fini: break
    return chemin, recomp
chemin, recomp = politique_gloutonne(Q)
print("Chemin appris :", chemin, "| retour (γ=0,95) :", round(retour(recomp, 0.95), 2))
verifier("Q7b", lambda: chemin[-1] == OBJECTIF and not (set(chemin) & CHOCS) and len(chemin) <= 9,
         "La politique apprise atteint l'objectif en évitant les chocs, par le chemin le plus court.")

**7c — Réflexion (à rédiger).** (i) Relancez `entrainer(gamma=0.5)` et `entrainer(eps=0.0)` : que change γ ? Pourquoi ε = 0 est-il dangereux ? (ii) Ce monde-grille est-il un bon modèle pour piloter un *vrai* portefeuille sur l'historique ? Appliquez la grille de décision de la session (simulateur validé ? récompense multi-critères ? échec acceptable ?).

*Votre réponse :* ...

## Q8 · Quiz de consolidation
Répondez par `A`, `B` ou `C`.

**Q8a.** Pourquoi un cluster n'a-t-il pas automatiquement un sens métier ?
- **A** — Parce que k-means est un algorithme instable.
- **B** — Parce qu'il regroupe par ressemblance géométrique : le sens se construit par interprétation et validation métier.
- **C** — Parce que les clusters sont toujours aléatoires.

**Q8b.** Comment évalue-t-on un modèle non supervisé, faute de cible ?
- **A** — Par l'AUC sur un jeu de test.
- **B** — Par la stabilité (rééchantillonnage), la couverture et l'utilité métier.
- **C** — Par le nombre de clusters : plus il y en a, mieux c'est.

**Q8c.** Un score d'anomalie très élevé signifie :
- **A** — Qu'il faut corriger automatiquement le contrat.
- **B** — Que le contrat est une fraude.
- **C** — Que le contrat est atypique : il mérite une revue experte, pas un verdict.

**Q8d.** Que maximise la PCA, et que perd-elle ?
- **A** — Elle maximise la variance conservée par axe, mais perd la signification directe des variables.
- **B** — Elle maximise la précision de prédiction, mais perd des observations.
- **C** — Elle maximise le nombre de clusters, mais perd la standardisation.

In [ ]:
rep_q8a, rep_q8b, rep_q8c, rep_q8d = "...", "...", "...", "..."
qcm("Q8a", rep_q8a, "8b59912273"); qcm("Q8b", rep_q8b, "bb2ba963f4"); qcm("Q8c", rep_q8c, "a4cebcf624"); qcm("Q8d", rep_q8d, "e4305b801f")

## Bilan

In [ ]:
bilan(["Q1", "Q2a", "Q2b", "Q2c", "Q3", "Q4", "Q5", "Q6a", "Q6b", "Q6c", "Q7a", "Q7b", "Q8a", "Q8b", "Q8c", "Q8d"])